# Preprocess Soccer Dataset

In [1]:
import csv
import random

## Read Data

In [4]:
soccer_data = [] #Our data as a list of lists (like Assignment 1)
#I will modify it's contents later

#This code opens the odd-not csv and turns it into ^list of list^
headers = []
first = True
with open('Soccer-Dataset.txt', 'r') as rawdata:
	atdata = False
	for eachline in rawdata:
		eachline = eachline.strip()
		if eachline.startswith('@ATTRIBUTE'): headers.append(eachline.split()[1])
		if eachline == '@DATA': atdata = True; continue
		if atdata == True:
			if first == True: first = False; soccer_data.append(headers)
			soccer_data.append(eachline.split(','))

## Data Parse Check

In [5]:
print("======================\n*Data Parse Check:\n", soccer_data[0], "\n", soccer_data[1], "\n======================\n")
number_of_samples = (len(soccer_data)-1)
winlosscount = [0, 0, 0]	#Contains the number of wins, draws, and losses: in that order
winlosspercent = [0, 0, 0]	#Same as ^above^ but percents in decimal (i.e. 0.624354)
for i in range(len(soccer_data)):
	if soccer_data[i][13] == "3": winlosscount[0] += 1
	if soccer_data[i][13] == "1": winlosscount[1] += 1
	if soccer_data[i][13] == "0": winlosscount[2] += 1
winlosspercent[0] = winlosscount[0]/number_of_samples
winlosspercent[1] = winlosscount[1]/number_of_samples
winlosspercent[2] = winlosscount[2]/number_of_samples

*Data Parse Check:
 ['Round', 'Date', 'Team_1', 'FT', 'HT', 'Team_2', 'Year', 'Country', 'FT_Team_1', 'FT_Team_2', 'HT_Team_1', 'HT_Team_2', 'GGD', 'Team_1_(pts)', 'Team_2_(pts)'] 
 ['1', "'(Sat) 19 Aug 1995 (W33)'", "'Aston Villa FC '", '3-1', '3-0', "'Manchester United FC '", '1995', 'ENG', '3', '1', '3', '0', '2', '3', '0'] 



## Home Team Win?

In [6]:
print("======================\n*Home Team Win?:\nWin:", str((round(winlosspercent[0] * 100, 2))) + "%", "\nDraw:", str((round(winlosspercent[1] * 100, 2))) + "%", "\nLoss:", str((round(winlosspercent[2] * 100, 2))) + "%", "\n======================\n")

#Here I check for missing values. This dataset has no missing values, so no extra preprocessing is needed here!
emptyvarcount = 0
for i in range(1, len(soccer_data)):
	for j in range(len(soccer_data[i])):
		if soccer_data[i][j] == '': emptyvarcount += 1

*Home Team Win?:
Win: 46.83% 
Draw: 26.45% 
Loss: 26.71% 



## Check For Missing Values

In [7]:
print("======================\nCheck For Missing Values:\n#+The number of missing/empty values is " + str(emptyvarcount) + "\n======================\n")

dupecount = 0
sofar = set()
for i in range(1, len(soccer_data)):
	check = (soccer_data[i][1], soccer_data[i][2], soccer_data[i][5])
	if check in sofar: dupecount += 1
	else: sofar.add(check)

Check For Missing Values:
#+The number of missing/empty values is 0



## Improved Dataset

In [8]:
def get_stats(history, n):
	last_few = history[-n:]
	points = 0
	goals = 0
	enemygoals = 0
	win = 0
	loss = 0
	homewin = 0
	for iterate_guy in last_few:
		points += iterate_guy['points']
		goals += iterate_guy['goals']
		enemygoals += iterate_guy['enemygoals']
		if iterate_guy['points']== 3: win += 1
		if iterate_guy['points']== 0: loss += 1
		if iterate_guy['points']== 3 and iterate_guy['hometeam'] == 1: homewin += 1
	
	return [points, goals, enemygoals, win, loss, homewin]

In [9]:
def last_ten_features(soccer_data_here):
	team_history = {}
	altered_data = [['Home_L3_Pts', 'Home_L3_Goals', 'Home_L3_EnemyGoals', 'Home_L3_Wins', 'Home_L3_Losses', 'Home_L3_HomeWins', 'Home_L5_Pts', 'Home_L5_Goals', 'Home_L5_EnemyGoals', 'Home_L5_Wins', 'Home_L5_Losses', 'Home_L5_HomeWins', 'Home_L10_Pts', 'Home_L10_Goals', 'Home_L10_EnemyGoals', 'Home_L10_Wins', 'Home_L10_Losses', 'Home_L10_HomeWins', 'Away_L3_Pts', 'Away_L3_Goals', 'Away_L3_EnemyGoals', 'Away_L3_Wins', 'Away_L3_Losses', 'Away_L3_HomeWins', 'Away_L5_Pts', 'Away_L5_Goals', 'Away_L5_EnemyGoals', 'Away_L5_Wins', 'Away_L5_Losses', 'Away_L5_HomeWins', 'Away_L10_Pts', 'Away_L10_Goals', 'Away_L10_EnemyGoals', 'Away_L10_Wins', 'Away_L10_Losses', 'Away_L10_HomeWins', 'Outlier?', 'Target_Did_Home_Win']]
	
	for i in range(1, len(soccer_data_here)):
		team1 = soccer_data_here[i][2]
		team2 = soccer_data_here[i][5]

		if team1 not in team_history: team_history[team1] = []
		if team2 not in team_history: team_history[team2] = []

		if len(team_history[team1]) >= 10 and len(team_history[team2]) >= 10:
			last3_team1 = get_stats(team_history[team1], 3)
			last5_team1 = get_stats(team_history[team1], 5)
			last10_team1 = get_stats(team_history[team1], 10)
			last3_team2 = get_stats(team_history[team2], 3)
			last5_team2 = get_stats(team_history[team2], 5)
			last10_team2 = get_stats(team_history[team2], 10)

			target = 0
			if int(soccer_data_here[i][13]) == 3: target = 1

			outlier = 0
			if int(soccer_data_here[i][8]) + int(soccer_data_here[i][9]) >= 8: outlier = 1

			altered_data.append(last3_team1 + last5_team1 + last10_team1 + last3_team2 + last5_team2 + last10_team2 + [outlier]  + [target])

		team_history[team1].append({'points': int(soccer_data_here[i][13]), 'goals': int(soccer_data_here[i][8]), 'enemygoals': int(soccer_data_here[i][9]), 'hometeam': 1})
		team_history[team2].append({'points': int(soccer_data_here[i][14]), 'goals': int(soccer_data_here[i][9]), 'enemygoals': int(soccer_data_here[i][8]), 'hometeam': 0})

	return altered_data

In [10]:
#This function changes the dataset: It should contain data about the
# last 3/5/10 games and home team win metric
better_soccer_data = last_ten_features(soccer_data)

## Feature Engineering Check

In [11]:
print("======================\n*Feature Engineering Check: ",
      len(better_soccer_data), "data samples,",
      (len(better_soccer_data[0])-1), "features\n",
      better_soccer_data[0], "\n",
      better_soccer_data[1], "\n======================\n")

*Feature Engineering Check:  42527 data samples, 37 features
 ['Home_L3_Pts', 'Home_L3_Goals', 'Home_L3_EnemyGoals', 'Home_L3_Wins', 'Home_L3_Losses', 'Home_L3_HomeWins', 'Home_L5_Pts', 'Home_L5_Goals', 'Home_L5_EnemyGoals', 'Home_L5_Wins', 'Home_L5_Losses', 'Home_L5_HomeWins', 'Home_L10_Pts', 'Home_L10_Goals', 'Home_L10_EnemyGoals', 'Home_L10_Wins', 'Home_L10_Losses', 'Home_L10_HomeWins', 'Away_L3_Pts', 'Away_L3_Goals', 'Away_L3_EnemyGoals', 'Away_L3_Wins', 'Away_L3_Losses', 'Away_L3_HomeWins', 'Away_L5_Pts', 'Away_L5_Goals', 'Away_L5_EnemyGoals', 'Away_L5_Wins', 'Away_L5_Losses', 'Away_L5_HomeWins', 'Away_L10_Pts', 'Away_L10_Goals', 'Away_L10_EnemyGoals', 'Away_L10_Wins', 'Away_L10_Losses', 'Away_L10_HomeWins', 'Outlier?', 'Target_Did_Home_Win'] 
 [3, 3, 3, 1, 2, 0, 7, 6, 4, 2, 2, 1, 17, 12, 8, 5, 3, 3, 2, 3, 5, 0, 1, 0, 2, 6, 10, 0, 3, 0, 9, 12, 15, 2, 5, 1, 0, 1] 



## Pearson Correlation

In [12]:
def pearson(feature, featuremean, target, targetmean):
	dividend = sum((float(feature[i]) - featuremean) * (float(target[i]) - targetmean) for i in range(len(feature)))
	featuredivisor = sum((float(feature[i]) - featuremean) ** 2 for i in range(len(feature)))
	targetdivisor = sum((float(target[i]) - targetmean) ** 2 for i in range(len(feature)))
	
	return dividend / (featuredivisor * targetdivisor) ** 0.5

In [13]:
def getmean(soccer_data_here): #I adapted this from my Assignment 1, pls don't shoot XD
	meanfirst = True
	mean = []
	missing = []
	for j in range(len(soccer_data_here[0])):
		mean.append(0)
		missing.append(0)

	for i in range(len(soccer_data_here)):
		if meanfirst == True:
			meanfirst = False
			continue
		for j in range(len(soccer_data_here[i])):
			if soccer_data_here[i][j] != '': mean[j] += float(soccer_data_here[i][j])
			else: missing[j] += 1
			
	for j in range(len(mean)):
		mean[j] = mean[j]/((len(soccer_data_here)-1)-missing[j])
		if j != len(mean)-1:
			mean [j] = round(mean[j], 4)
		#print(mean[j])
	#now I have the means XD
	return mean

In [14]:
# Pearson Correlation code adapted from my Assignment 1 (Kurtis Volker)
mean = getmean(better_soccer_data)
columns = []

for j in range(len(better_soccer_data[0])):
	columns.append([])
		
for i in range(len(better_soccer_data)):
	if i == 0: continue
	for j in range(len(better_soccer_data[i])):
		columns[j].append(better_soccer_data[i][j])

Pcorrelations = []
for i in range(len(columns)):
	Pcorrelations.append(pearson(columns[i], mean[i], columns[-1], mean[-1]))

print("======================\nPearson Correlations:\n")
for i, correlation in enumerate(Pcorrelations):
    print(f"Feature {i}: {correlation:.4f}")
print("\n======================\n")

Pearson Correlations:

Feature 0: 0.1055
Feature 1: 0.1052
Feature 2: -0.0715
Feature 3: 0.0992
Feature 4: -0.0892
Feature 5: 0.0646
Feature 6: 0.1310
Feature 7: 0.1295
Feature 8: -0.0915
Feature 9: 0.1234
Feature 10: -0.1136
Feature 11: 0.0876
Feature 12: 0.1659
Feature 13: 0.1582
Feature 14: -0.1256
Feature 15: 0.1582
Feature 16: -0.1474
Feature 17: 0.1207
Feature 18: -0.1051
Feature 19: -0.1023
Feature 20: 0.0754
Feature 21: -0.0986
Feature 22: 0.0893
Feature 23: -0.0771
Feature 24: -0.1284
Feature 25: -0.1209
Feature 26: 0.0942
Feature 27: -0.1214
Feature 28: 0.1106
Feature 29: -0.0939
Feature 30: -0.1551
Feature 31: -0.1443
Feature 32: 0.1188
Feature 33: -0.1480
Feature 34: 0.1376
Feature 35: -0.1159
Feature 36: 0.0164
Feature 37: 1.0000




## Feature Selection

In [17]:
feature_indices = list(range(len(better_soccer_data[0]) - 1))  # exclude target column

kept_indices   = [i for i in feature_indices if abs(Pcorrelations[i]) >= threshold]
dropped_indices = [i for i in feature_indices if abs(Pcorrelations[i]) <  threshold]

# Build the new dataset: kept feature columns + target column (always last)
target_col_idx = len(better_soccer_data[0]) - 1
selected_cols  = kept_indices + [target_col_idx]

selected_data = []
for i, row in enumerate(better_soccer_data):
    selected_data.append([row[j] for j in selected_cols])

print("======================")
print("*Feature Selection:")
print(f"  Threshold: |r| >= {threshold}")
print(f"  Features dropped ({len(dropped_indices)}):")
for idx in dropped_indices:
    print(f"    Feature {idx} ({better_soccer_data[0][idx]}): r = {Pcorrelations[idx]:.4f}")
print(f"  Features kept: {len(kept_indices)}")
print(f"  New dataset shape: {len(selected_data)-1} samples x {len(selected_data[0])-1} features")
print("\n  New headers:", selected_data[0])
print("======================")


*Feature Selection:
  Threshold: |r| >= 0.05
  Features dropped (1):
    Feature 36 (Outlier?): r = 0.0164
  Features kept: 36
  New dataset shape: 42526 samples x 36 features

  New headers: ['Home_L3_Pts', 'Home_L3_Goals', 'Home_L3_EnemyGoals', 'Home_L3_Wins', 'Home_L3_Losses', 'Home_L3_HomeWins', 'Home_L5_Pts', 'Home_L5_Goals', 'Home_L5_EnemyGoals', 'Home_L5_Wins', 'Home_L5_Losses', 'Home_L5_HomeWins', 'Home_L10_Pts', 'Home_L10_Goals', 'Home_L10_EnemyGoals', 'Home_L10_Wins', 'Home_L10_Losses', 'Home_L10_HomeWins', 'Away_L3_Pts', 'Away_L3_Goals', 'Away_L3_EnemyGoals', 'Away_L3_Wins', 'Away_L3_Losses', 'Away_L3_HomeWins', 'Away_L5_Pts', 'Away_L5_Goals', 'Away_L5_EnemyGoals', 'Away_L5_Wins', 'Away_L5_Losses', 'Away_L5_HomeWins', 'Away_L10_Pts', 'Away_L10_Goals', 'Away_L10_EnemyGoals', 'Away_L10_Wins', 'Away_L10_Losses', 'Away_L10_HomeWins', 'Target_Did_Home_Win']


## Range Normalization (Min-Max 0 to 1)

In [18]:
n_features = len(selected_data[0]) - 1  # exclude target

# Compute per-feature min and max (skip header row)
col_min = [float('inf')]  * n_features
col_max = [float('-inf')] * n_features

for i in range(1, len(selected_data)):
    for j in range(n_features):
        val = float(selected_data[i][j])
        if val < col_min[j]: col_min[j] = val
        if val > col_max[j]: col_max[j] = val

# Build normalized dataset (header row stays as strings)
normalized_data = [selected_data[0][:]]  # copy header

for i in range(1, len(selected_data)):
    norm_row = []
    for j in range(n_features):
        val   = float(selected_data[i][j])
        denom = col_max[j] - col_min[j]
        norm_row.append(round((val - col_min[j]) / denom, 6) if denom != 0 else 0.0)
    norm_row.append(selected_data[i][-1])  # preserve original target
    normalized_data.append(norm_row)

print("======================")
print("*Range Normalization (Min-Max [0, 1]):")
print(f"  Features normalized: {n_features}  |  Target column preserved")
print(f"  Sample normalized row [0]: {normalized_data[1]}")
print("\n  Per-feature ranges (min → max):")
for j in range(n_features):
    print(f"    {selected_data[0][j]:30s}  min={col_min[j]:.1f}  max={col_max[j]:.1f}")
print("======================")


*Range Normalization (Min-Max [0, 1]):
  Features normalized: 36  |  Target column preserved
  Sample normalized row [0]: [0.333333, 0.15, 0.176471, 0.333333, 0.666667, 0.0, 0.466667, 0.206897, 0.173913, 0.4, 0.4, 0.25, 0.566667, 0.266667, 0.210526, 0.5, 0.3, 0.5, 0.222222, 0.166667, 0.277778, 0.0, 0.333333, 0.0, 0.133333, 0.24, 0.434783, 0.0, 0.6, 0.0, 0.3, 0.261905, 0.416667, 0.2, 0.5, 0.166667, 1]

  Per-feature ranges (min → max):
    Home_L3_Pts                     min=0.0  max=9.0
    Home_L3_Goals                   min=0.0  max=20.0
    Home_L3_EnemyGoals              min=0.0  max=17.0
    Home_L3_Wins                    min=0.0  max=3.0
    Home_L3_Losses                  min=0.0  max=3.0
    Home_L3_HomeWins                min=0.0  max=3.0
    Home_L5_Pts                     min=0.0  max=15.0
    Home_L5_Goals                   min=0.0  max=29.0
    Home_L5_EnemyGoals              min=0.0  max=23.0
    Home_L5_Wins                    min=0.0  max=5.0
    Home_L5_Losses        